In [ ]:
# Install dependencies
!pip install --quiet detoxify pandas torch torchvision torchaudio

# Import libraries
import pandas as pd
from detoxify import Detoxify
import math, os
from google.colab import files

# Upload the CSV file (choose from your computer)
uploaded = files.upload()
csv_path = next(iter(uploaded.keys()))  # uploaded file name

df = pd.read_csv(csv_path)
assert 'no_names' in df.columns, "The file must contain the 'no_names' column."

# Load the multilingual model
model = Detoxify('multilingual')

# Set parameters
BATCH_SIZE = 250  # number of comments per batch
OUT_DIR = "output_batches"

os.makedirs(OUT_DIR, exist_ok=True)
n = len(df)
n_batches = math.ceil(n / BATCH_SIZE)

print(f"Total comments: {n}")
print(f"Creating {n_batches} batches of {BATCH_SIZE} comments each.\n")

# Loop through each batch
for i in range(n_batches):
    start = i * BATCH_SIZE
    end = min((i + 1) * BATCH_SIZE, n)
    print(f"Processing batch {i+1}/{n_batches} -> rows {start}-{end-1}")

    subset = df.iloc[start:end].copy()
    texts = subset['no_names'].fillna('').astype(str).tolist()

    # Analyze with Detoxify
    results = model.predict(texts)

    # Add result columns
    for k in results:
        subset[k] = results[k]

    # Save batch
    out_path = os.path.join(OUT_DIR, f"batch_{start}_{end-1}.csv")
    subset.to_csv(out_path, index=False)

    print(f"Saved: {out_path}\n")

print("All batches completed!")

# (Optional) Create a ZIP archive to download them all together
import shutil
shutil.make_archive("toxicity_batches", 'zip', OUT_DIR)
files.download("toxicity_batches.zip")

This part of the code puts all your data back together by taking all those small batch files you just created and merging them into one big dataset. After that, it looks at the toxicity scores from the model. Since those scores are just decimals between 0 and 1, the script groups them into specific buckets or "bins" to make things easier to read. Finally, it counts how many comments landed in each bucket and draws a bar chart.



In [ ]:
# 1. Import libraries

import matplotlib.pyplot as plt


# 2. Upload ALL CSV files
uploaded = files.upload()

dfs = []

for filename in uploaded.keys():
    print(f"Loaded: {filename}")
    df = pd.read_csv(filename)

    assert 'toxicity' in df.columns, f"'{filename}' does not contain the 'toxicity' column"
    dfs.append(df)

# 3. Merge all batches
all_data = pd.concat(dfs, ignore_index=True)
print(f"Total comments: {len(all_data)}")

# 4. Create bins for the bar chart
bins = [0, 0.05, 0.1, 0.2, 0.4, 0.6, 0.8, 1.0]
labels = [
    '0–0.05',
    '0.05–0.1',
    '0.1–0.2',
    '0.2–0.4',
    '0.4–0.6',
    '0.6–0.8',
    '0.8–1.0'
]

all_data['toxicity_bin'] = pd.cut(
    all_data['toxicity'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# 5. Bar chart
counts = all_data['toxicity_bin'].value_counts().sort_index()

plt.figure()
# Optional: Add color to the bars for better visualization
plt.bar(counts.index.astype(str), counts.values, color='skyblue', edgecolor='black')
plt.title('Toxicity Distribution')
plt.xlabel('Level of Toxicity')
plt.ylabel('Number of Comments')
plt.xticks(rotation=45)
plt.tight_layout()



plt.show()

In [ ]:
# 1. Import libraries
import pandas as pd
from google.colab import files

# 2. Upload ALL CSV files
uploaded = files.upload()

dfs = []
for fname in uploaded.keys():
    df = pd.read_csv(fname)
    assert 'toxicity' in df.columns, f"'{fname}' is missing the 'toxicity' column"
    assert 'no_names' in df.columns, f"'{fname}' is missing the 'no_names' column"
    dfs.append(df)

# 3. Merge the batches
all_data = pd.concat(dfs, ignore_index=True)

print(f"Total comments: {len(all_data)}")

# 4. Define toxicity ranges
ranges = {
    "LOW (0-0.05)": (0.0, 0.05),
    "MEDIUM (0.2-0.4)": (0.2, 0.4),
    "HIGH (0.6-0.8)": (0.6, 0.8),
    "VERY HIGH (0.8-1.0)": (0.8, 1.0)
}

N = 5  # examples per range

# 5. Print examples
for label, (low, high) in ranges.items():
    print("\n" + "-"*80)
    print(label)
    print("-"*80)

    subset = all_data[
        (all_data['toxicity'] >= low) &
        (all_data['toxicity'] < high)
    ]

    if subset.empty:
        print("No comments found in this range")
        continue

    samples = subset.sample(min(N, len(subset)), random_state=42)

    for i, row in samples.iterrows():
        print(f"\nToxicity: {row['toxicity']:.3f}")
        print(row['no_names'])

This piece of code grabs all your smaller files and stitches them together and acts like a filter. It sets up four different buckets—from "low" to "very high" toxicity—and then digs into your dataset to pull out up to 5 random comments for each bucket. It then prints them out on your screen along with their exact score. This is a way to do a quick manual check and see if the AI actually did a good job labeling the bad comments.